In [4]:
import json
import re
import time
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any

import pandas as pd
from pdf2image import convert_from_path
import mimetypes

from google import genai
from google.genai.types import HttpOptions, Part, GenerateContentConfig
from google.oauth2.credentials import Credentials

import logging

# ============================================================
# LOGGING
# ============================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

# ============================================================
# CONFIG (PUT YOUR TOKEN HERE)
# ============================================================
base_url = "https://vertexai.prod.ai-gateway.quantumblack.com/7a4f3d63-b5db-4d5b-8076-8f114d1f14f7/"
access_token = "eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJhZXNKN2kxNGNidnVuTU40MTJrOU5yZ2ROeENhTlJudTNPbC1TU08ycFlJIn0.eyJleHAiOjE3NjUxNjUzMzEsImlhdCI6MTc2NTE2MzUzMiwiYXV0aF90aW1lIjoxNzY1MTYzNTMxLCJqdGkiOiJhOGRjM2I4My1lODg1LTRiYzMtOGI3Ny00OWZmOGJiODM5NjIiLCJpc3MiOiJodHRwczovL2F1dGgubWNraW5zZXkuaWQvYXV0aC9yZWFsbXMvciIsImF1ZCI6ImJjZDIzNzI4LTNkMjctNDQ3Yy1hMGE5LWVhY2FmMzkzYTZmNSIsInN1YiI6IjI0NDRiYzZjLTAwMzctNGIyZS1hYzI3LWZjNTlhNTkxNTM2NiIsInR5cCI6IklEIiwiYXpwIjoiYmNkMjM3MjgtM2QyNy00NDdjLWEwYTktZWFjYWYzOTNhNmY1Iiwic2Vzc2lvbl9zdGF0ZSI6ImNlM2U1NDE5LTlkNGItNGY2Ny05NDkxLTllMTc0OTZmYmZhNyIsImF0X2hhc2giOiJzTktGb2I5ZS1sWE9DZUFObk1Uc25BIiwibmFtZSI6IlVnYW5kaGFyIFZhZGRpIiwiZ2l2ZW5fbmFtZSI6IlVnYW5kaGFyIiwiZmFtaWx5X25hbWUiOiJWYWRkaSIsInByZWZlcnJlZF91c2VybmFtZSI6IjE1ZDhiYmNkMmMzNTNmYWUiLCJlbWFpbCI6IlVnYW5kaGFyX1ZhZGRpQG1ja2luc2V5LmNvbSIsImFjciI6IjEiLCJzaWQiOiJjZTNlNTQxOS05ZDRiLTRmNjctOTQ5MS05ZTE3NDk2ZmJmYTciLCJlbWFpbF92ZXJpZmllZCI6dHJ1ZSwiZm1ubyI6IjM0NzI3NiIsImdyb3VwcyI6WyI3YTRmM2Q2My1iNWRiLTRkNWItODA3Ni04ZjExNGQxZjE0ZjciLCJBbGwgRmlybSBVc2VycyJdfQ.LmOdwZB5ZaAoGljWD5_VDKfAkiXu_ZnJ3MZB9O8WI7SELSIjsWO6imlckcvWyxY2WW7XnhitHwQ8TbgXINnjP-BcPxz3RmxD023bEu70WnBqoVCzKKHGMnbUUpsP3Dp6DDZPvVv96s1W5H5A2GNy7mRWvJazlNhSIe7NKLqRuzPPjuj_weNd5Sal175gzNcTAj4T2ZIuR3S3J9r_nosdDbEKOeLbA2HIfHbwhr8iC8LPVRxl1acLACeYEr0PTMYEtDvOMYrPl9-AzF9LCUE0yPlzNVHmdfQQHSKIfWV28IHm1wFp0Q4UImOBvngI2DYJdjWI1nfWMVAAH3fTf3LKyQ"
credentials = Credentials(access_token)

client = genai.Client(
    http_options=HttpOptions(
        api_version="v1",
        base_url=base_url,
    ),
    vertexai=True,
    project="aigateway",
    location="global",
    credentials=credentials,
)

MODEL_NAME = "gemini-2.5-pro"
logger.info("Simple Engineering Table → Excel extractor ready.")


# ============================================================
# PROMPT (STRICT, BUT SHORT)
# ============================================================
def get_primary_extraction_prompt() -> str:
    return """
You are an Engineering Table Extraction AI.

GOAL:
- Detect structured tables in the drawing (rows + columns).
- Use the ACTUAL TABLE TITLE printed near each table as its name.

TABLE TITLE RULES (VERY IMPORTANT):
- For each table, look for a clear title/heading:
  - Usually ALL CAPS or bold, directly ABOVE or INSIDE the table.
  - Examples: "BILL OF MATERIALS", "REVISION HISTORY", "CABLE SCHEDULE",
    "CONNECTOR TABLE", "LEGEND", "WIRE LIST", "SPECIFICATIONS".
- Use that exact text (trimmed) as the key in the "tables" object and as the "title" field.
- If there is no clear title:
  - Use a generic name like "Untitled Table", "Untitled Table 1", "Untitled Table 2", etc.
- DO NOT invent new technical names. Only use what is printed.

WHAT TO EXTRACT:
- ONLY data that is clearly visible in structured tables (rows + columns).
- DO NOT guess, infer, or invent any values.
- Copy text EXACTLY as shown (keep case, symbols, units).
- If a cell is blank, use null.
- If text is unreadable, use "ILLEGIBLE".
- Under-extraction is OK. Hallucination is NOT OK.

TABLE TYPES TO EXTRACT (if present):
- Bill of Materials / Parts List
- Revision / Change History
- Technical Specification / Data Tables
- Schedules (wire lists, cable tables, equipment lists, etc.)
- Legend / Symbol tables
- Any other grid-like table with headers and rows

OUTPUT FORMAT (STRICT JSON, NO MARKDOWN):

{
  "extraction_metadata": {
    "confidence_level": "high" | "medium" | "low",
    "tables_found": <int>,
    "quality_issues": ["text ...", "text ..."]
  },
  "tables": {
    "<Exact table title 1>": {
      "title": "<Exact table title 1>",
      "headers": ["Col1", "Col2", ...],
      "rows": [
        ["v11", "v12", ...],
        ["v21", null, ...]
      ],
      "row_count": <int>,
      "column_count": <int>
    },
    "<Exact table title 2>": {
      "title": "<Exact table title 2>",
      "headers": [...],
      "rows": [...],
      "row_count": <int>,
      "column_count": <int>
    }
  }
}

RULES:
- "tables" is a dict:
  - KEY = exact table title text from the drawing (or "Untitled Table X" if no title).
  - VALUE = table object with "title", "headers", "rows", "row_count", "column_count".
- First row is NOT repeated in "rows".
- Every row in "rows" must have the same length as "headers".
- If no tables exist, return:

{
  "extraction_metadata": {
    "confidence_level": "high",
    "tables_found": 0,
    "quality_issues": ["No tables detected on this page"]
  },
  "tables": {}
}
"""


# ============================================================
# MODEL CALL + JSON CLEANING
# ============================================================
def clean_json_response(raw_text: str) -> str:
    """Remove code fences and trim to outermost { ... } if needed."""
    if not raw_text:
        return "{}"

    if "```json" in raw_text:
        raw_text = raw_text.split("```json", 1)[1]
        raw_text = raw_text.split("```", 1)[0]
    elif "```" in raw_text:
        raw_text = raw_text.split("```", 1)[1]
        raw_text = raw_text.split("```", 1)[0]

    raw_text = raw_text.strip()

    if not (raw_text.startswith("{") and raw_text.endswith("}")):
        start = raw_text.find("{")
        end = raw_text.rfind("}")
        if start != -1 and end != -1 and end > start:
            raw_text = raw_text[start : end + 1]

    return raw_text.strip() or "{}"


def call_model_on_image(image_path: str) -> Dict[str, Any]:
    """Call Gemini once for this image and return parsed JSON (or empty structure)."""
    logger.info(f"Calling model on: {image_path}")

    with open(image_path, "rb") as f:
        image_bytes = f.read()

    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None:
        mime_type = "image/png"

    image_part = Part.from_bytes(data=image_bytes, mime_type=mime_type)

    resp = client.models.generate_content(
        model=MODEL_NAME,
        contents=[get_primary_extraction_prompt(), image_part],
        config=GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0.1,
        ),
    )

    raw = resp.text or "{}"
    cleaned = clean_json_response(raw)

    try:
        data = json.loads(cleaned)
    except json.JSONDecodeError as e:
        logger.error(f"JSON decode error: {e}")
        return {
            "extraction_metadata": {
                "confidence_level": "low",
                "tables_found": 0,
                "quality_issues": [f"json_parse_error: {str(e)}"],
            },
            "tables": {},
        }

    if isinstance(data, list):
        data = next(
            (item for item in data if isinstance(item, dict) and "tables" in item),
            data[0] if data else {},
        )

    if not isinstance(data, dict):
        logger.error(f"Top-level response is not dict (got {type(data)})")
        return {
            "extraction_metadata": {
                "confidence_level": "low",
                "tables_found": 0,
                "quality_issues": ["model_response_not_dict"],
            },
            "tables": {},
        }

    if "tables" not in data or not isinstance(data["tables"], dict):
        data["tables"] = {}

    if "extraction_metadata" not in data or not isinstance(
        data["extraction_metadata"], dict
    ):
        data["extraction_metadata"] = {
            "confidence_level": "unknown",
            "tables_found": len(data["tables"]),
            "quality_issues": [],
        }

    return data


# ============================================================
# LOCAL STRUCTURE FIX
# ============================================================
def fix_and_validate_tables(data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Ensure each table has:
      - headers: list
      - rows: list of lists
      - row_count, column_count consistent
    Drop tables that are clearly broken.
    """
    tables = data.get("tables", {})
    fixed_tables: Dict[str, Any] = {}

    for name, tbl in tables.items():
        if not isinstance(tbl, dict):
            logger.warning(f"Skipping table '{name}' (not a dict).")
            continue

        headers = tbl.get("headers", [])
        rows = tbl.get("rows", [])

        if not isinstance(headers, list) or len(headers) == 0:
            logger.warning(f"Skipping table '{name}' (missing/empty headers).")
            continue

        col_count = len(headers)

        if not isinstance(rows, list):
            logger.warning(f"Table '{name}' rows not a list, treating as empty.")
            rows = []

        fixed_rows: List[List[Any]] = []
        for r_idx, row in enumerate(rows):
            if not isinstance(row, list):
                row = [row]

            if len(row) < col_count:
                row = row + [None] * (col_count - len(row))
            elif len(row) > col_count:
                row = row[:col_count]

            norm_row = [None if (c == "" or c is None) else c for c in row]
            fixed_rows.append(norm_row)

        fixed_tables[name] = {
            "headers": headers,
            "rows": fixed_rows,
            "row_count": len(fixed_rows),
            "column_count": col_count,
        }

    data["tables"] = fixed_tables
    meta = data.get("extraction_metadata", {})
    meta["tables_found"] = len(fixed_tables)
    data["extraction_metadata"] = meta
    return data


# ============================================================
# EXCEL EXPORT ONLY
# ============================================================
def export_tables_to_excel(all_tables: Dict[str, Any], excel_path: str) -> None:
    """all_tables: table_name -> {headers, rows}"""
    logger.info(f"Writing Excel: {excel_path} (tables: {len(all_tables)})")

    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        used_sheet_names = set()

        for name, tbl in all_tables.items():
            headers = tbl.get("headers", [])
            rows = tbl.get("rows", [])

            df = pd.DataFrame(rows, columns=headers)

            # Clean sheet name
            clean_name = re.sub(r'[\\/*?:\[\]]', "", str(name))[:31]
            if not clean_name:
                clean_name = "Table"

            original = clean_name
            k = 1
            while clean_name in used_sheet_names:
                suffix = f"_{k}"
                clean_name = (original[: 31 - len(suffix)] + suffix)
                k += 1
            used_sheet_names.add(clean_name)

            df.to_excel(writer, sheet_name=clean_name, index=False)

    logger.info("Excel export complete.")


# ============================================================
# MAIN PIPELINE – EXCEL ONLY
# ============================================================
def process_file_to_excel(file_path: str) -> None:
    """
    - If PDF: convert each page to image (300 DPI)
    - If image: process directly
    - For each page, extract tables via Gemini
    - Combine all tables into ONE Excel file
    - No JSON returned to caller – output is Excel only
    """
    path = Path(file_path)
    logger.info("=" * 60)
    logger.info(f"Processing file: {path}")
    logger.info("=" * 60)

    if not path.exists():
        logger.error("ERROR: File does not exist.")
        return

    ext = path.suffix.lower()
    temp_images: List[str] = []

    try:
        # 1) PDF → per-page images
        if ext == ".pdf":
            logger.info("Converting PDF to images (300 DPI)...")
            pages = convert_from_path(str(path), dpi=300)
            if not pages:
                logger.error("PDF conversion returned 0 pages.")
                return

            for idx, page in enumerate(pages, start=1):
                img_path = f"temp_page_{idx}_{int(time.time())}.png"
                page.save(img_path, "PNG")
                temp_images.append(img_path)
                logger.info(f"  Saved page {idx} as {img_path}")
        else:
            logger.info("Processing as single image file.")
            temp_images = [str(path)]

        all_tables: Dict[str, Any] = {}

        # 2) Process each page/image
        for i, img_path in enumerate(temp_images, start=1):
            logger.info(f"\n--- Page/Image {i}/{len(temp_images)} ---")
            result = call_model_on_image(img_path)
            result = fix_and_validate_tables(result)

            meta = result.get("extraction_metadata", {})
            logger.info(
                f"  tables_found={meta.get('tables_found', 0)}, "
                f"confidence={meta.get('confidence_level', 'unknown')}"
            )

            for tname, tdata in result.get("tables", {}).items():
                if len(temp_images) > 1:
                    base_name = f"p{i}_{tname}"
                else:
                    base_name = tname

                final_name = base_name
                c = 1
                while final_name in all_tables:
                    final_name = f"{base_name}_{c}"
                    c += 1

                all_tables[final_name] = tdata
                logger.info(f"  ✓ captured table: {final_name}")

        # 3) Excel only
        if not all_tables:
            logger.warning("No tables found in any page. Excel not created.")
            return

        excel_name = f"{path.stem}_ExtractedTables.xlsx"
        export_tables_to_excel(all_tables, excel_name)
        logger.info(f"\nSUCCESS: Excel created -> {excel_name}")

    finally:
        # 4) Cleanup temp images
        if ext == ".pdf":
            for img in temp_images:
                try:
                    Path(img).unlink()
                except Exception:
                    pass
            logger.info("Temporary images cleaned up.")


# ============================================================
# ENTRY POINT EXAMPLE
# ============================================================
if __name__ == "__main__":
    print("\nSimple Engineering Table → Excel Extractor")
    print("Usage:")
    print("  process_file_to_excel('drawing.pdf')")
    print("  process_file_to_excel('Master.png')")


2025-12-08 03:31:35,847 - INFO - Simple Engineering Table → Excel extractor ready.



Simple Engineering Table → Excel Extractor
Usage:
  process_file_to_excel('drawing.pdf')
  process_file_to_excel('Master.png')


In [6]:
#from table_extractor_excel import process_file_to_excel

process_file_to_excel("Master.png")
# or
#process_file_to_excel("your_drawing.pdf")


2025-12-08 03:31:52,015 - INFO - ============================================================
2025-12-08 03:31:52,017 - INFO - Processing file: Master.png
2025-12-08 03:31:52,019 - INFO - ============================================================
2025-12-08 03:31:52,021 - INFO - Processing as single image file.
2025-12-08 03:31:52,023 - INFO - 
--- Page/Image 1/1 ---
2025-12-08 03:31:52,024 - INFO - Calling model on: Master.png
2025-12-08 03:31:52,026 - INFO - AFC is enabled with max remote calls: 10.
2025-12-08 03:34:31,284 - INFO - HTTP Request: POST https://vertexai.prod.ai-gateway.quantumblack.com/7a4f3d63-b5db-4d5b-8076-8f114d1f14f7/v1/projects/aigateway/locations/global/publishers/google/models/gemini-2.5-pro:generateContent "HTTP/1.1 200 OK"
2025-12-08 03:34:31,288 - INFO -   tables_found=16, confidence=high
2025-12-08 03:34:31,289 - INFO -   ✓ captured table: Table_1
2025-12-08 03:34:31,290 - INFO -   ✓ captured table: Table_2
2025-12-08 03:34:31,291 - INFO -   ✓ captured tab